# THS-II EMS — Self-Contained PPO Training Notebook (Colab + GPU)

Trains a PPO energy-management agent on the THS-II hybrid powertrain environment.

**This notebook is fully self-contained.** Upload *only this `.ipynb`* to Colab — no `git clone`,
no file uploads, no Drive mount. The project's source files (`modeling.py`, the `env/` package,
and the three drive-cycle CSVs) are embedded below as a compressed blob and written to
`/content/pfa/` at runtime.

**Before you start — enable the GPU:** `Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU`.

## 1. Install dependencies

Colab already ships a CUDA-enabled `torch`, so we do **not** reinstall it. We only add Stable-Baselines3 and pin Gymnasium.

In [ ]:
%pip install -q "stable-baselines3==2.8.0" "gymnasium==0.29.1"

## 2. Confirm the GPU is visible

In [ ]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU - enable it via Runtime > Change runtime type > GPU.")

## 3. Unpack the embedded project files

Decodes the gzip+base64 bundle and writes `modeling.py`, `env/`, and the drive-cycle CSVs to `PROJECT_ROOT`, then puts that directory on `sys.path`.

In [ ]:
import base64, io, tarfile, sys
from pathlib import Path

PROJECT_ROOT = Path("/content/pfa")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# gzip+base64 tarball of: modeling.py, env/__init__.py, env/ths_env.py,
# env/drive_cycles/{WLTC,FTP75,US06}.csv
BUNDLE_B64 = "..."

raw = base64.b64decode(BUNDLE_B64)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:gz") as tar:
    tar.extractall(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Extracted to", PROJECT_ROOT)
for p in sorted(PROJECT_ROOT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(PROJECT_ROOT))

## 4. Imports and training helpers

The `THSEnv` Gymnasium wrapper comes from the unpacked package. The small vec-env / schedule helpers (normally in `training/train_ppo.py`) are inlined here so nothing else needs unpacking.

In [ ]:
import random
from typing import Callable

import numpy as np
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.vec_env import (
    DummyVecEnv, SubprocVecEnv, VecMonitor, VecNormalize,
)

import env.ths_env as ths_env_mod
from env.ths_env import THSEnv

ALL_CYCLES = ("WLTC", "FTP75", "US06")
ENV_DIR = Path(ths_env_mod.__file__).resolve().parent


class MultiCycleEnv(THSEnv):
    """Randomly picks a drive cycle on each reset for better generalisation."""

    def __init__(self, seed: int = 0, route_cache=None):
        self._rng = random.Random(seed)
        super().__init__(cycle=self._rng.choice(ALL_CYCLES), route_cache=route_cache)

    def reset(self, *, seed=None, options=None):
        self.cycle_name = self._rng.choice(ALL_CYCLES)
        self.cycle_path = ENV_DIR / "drive_cycles" / self.CYCLE_FILES[self.cycle_name]
        return super().reset(seed=seed, options=options)


def _make_single_env(cycle, seed, route_cache=None):
    def _init():
        if cycle == "all":
            e = MultiCycleEnv(seed=seed, route_cache=route_cache)
        else:
            e = THSEnv(cycle, route_cache=route_cache)
        e.reset(seed=seed)
        return Monitor(e)
    return _init


def make_vec_env(cycle, seed, n_envs, route_cache=None):
    thunks = [_make_single_env(cycle, seed + i, route_cache) for i in range(n_envs)]
    if n_envs == 1:
        return DummyVecEnv(thunks)
    return SubprocVecEnv(thunks, start_method="spawn")


def linear_schedule(initial: float, final_frac: float) -> Callable[[float], float]:
    final = initial * final_frac
    def _schedule(progress_remaining: float) -> float:
        return final + (initial - final) * progress_remaining
    return _schedule

## 5. Configuration

Mirrors the CLI arguments of the original `train_ppo.py`. Lower `TOTAL_TIMESTEPS` for a quick smoke run; the script default is `2_000_000`.

**Colab note on `N_ENVS`:** `SubprocVecEnv` (spawn) is used when `N_ENVS > 1`. Free Colab has ~2 vCPUs, so env stepping is the bottleneck and the GPU mainly accelerates policy updates. If subprocess workers hang, set `N_ENVS = 1` (uses `DummyVecEnv`).

In [ ]:
CYCLE            = "all"        # "WLTC" | "FTP75" | "US06" | "all" (randomised per episode)
SEED             = 42
TOTAL_TIMESTEPS  = 2_000_000   # reduce (e.g. 100_000) for a fast trial run
N_ENVS           = 4           # parallel rollout workers (try 1 if subprocs hang on Colab)
LEARNING_RATE    = 3e-4
LR_FINAL_FRAC    = 0.1
N_STEPS          = 1024        # rollout length per env
BATCH_SIZE       = 256
N_EPOCHS         = 10
GAMMA            = 0.99
GAE_LAMBDA       = 0.95
CLIP_RANGE       = 0.2
ENT_COEF         = 0.01
USE_VECNORMALIZE = True
EVAL_FREQ        = 25_000      # env-steps between evaluations
N_EVAL_EPISODES  = 3
DEVICE           = "cpu"      # MLP PPO is faster on CPU; GPU adds transfer overhead (SB3 advises CPU)
ROUTE_CACHE      = None

MODEL_DIR        = PROJECT_ROOT / "models"
TENSORBOARD_LOG  = PROJECT_ROOT / "tb_logs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
TENSORBOARD_LOG.mkdir(parents=True, exist_ok=True)

set_random_seed(SEED)
print("Training device:", DEVICE)

## 6. Build the training and evaluation environments

Observations are already normalised inside `THSEnv`, so `VecNormalize` only whitens the shaped reward. The eval env spans all three drive cycles and reports raw episode reward.

In [ ]:
# Training env
train_env = make_vec_env(CYCLE, SEED, N_ENVS, ROUTE_CACHE)
train_env = VecMonitor(train_env)
if USE_VECNORMALIZE:
    train_env = VecNormalize(train_env, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env: all three cycles for a representative metric
eval_env = make_vec_env("all", SEED + 10_000, max(1, N_ENVS // 2), ROUTE_CACHE)
eval_env = VecMonitor(eval_env)
if USE_VECNORMALIZE:
    eval_env = VecNormalize(eval_env, norm_obs=False, norm_reward=True, clip_reward=10.0)
    eval_env.training = False       # freeze running stats during eval
    eval_env.norm_reward = False    # report raw episode reward

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(MODEL_DIR),
    log_path=str(MODEL_DIR / "eval_logs"),
    eval_freq=max(EVAL_FREQ // N_ENVS, 1),
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    render=False,
)

## 7. Build the PPO model

128x128 Tanh MLP for both policy and value heads, with linear-decay schedules on the learning rate and clip range and a small entropy bonus to keep mode exploration alive early.

In [ ]:
policy_kwargs = {
    "net_arch": dict(pi=[128, 128], vf=[128, 128]),
    "activation_fn": nn.Tanh,
}

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=linear_schedule(LEARNING_RATE, LR_FINAL_FRAC),
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_range=linear_schedule(CLIP_RANGE, LR_FINAL_FRAC),
    ent_coef=ENT_COEF,
    policy_kwargs=policy_kwargs,
    tensorboard_log=str(TENSORBOARD_LOG),
    seed=SEED,
    device=DEVICE,
    verbose=1,
)

cycle_label = CYCLE if CYCLE != "all" else "WLTC+FTP75+US06"
print(f"Training PPO on {cycle_label} for {TOTAL_TIMESTEPS:,} timesteps on {DEVICE}")
print(f"Net 128x128 (pi/vf) | {N_ENVS} envs | rollout={N_STEPS * N_ENVS} | batch={BATCH_SIZE} "
      f"| ent={ENT_COEF} | vecnorm={USE_VECNORMALIZE}")
print(f"Eval on all cycles every {EVAL_FREQ:,} env-steps ({N_EVAL_EPISODES} eps)")

## 8. (Optional) Live training curves with TensorBoard

Run this *before* training to watch reward/loss curves update in place.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$TENSORBOARD_LOG"

## 9. Train

This is the long-running cell. Progress and eval metrics stream below.

In [ ]:
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=eval_callback,
    tb_log_name=f"ppo_{CYCLE.lower()}",
    reset_num_timesteps=True,
)

## 10. Save the trained agent

Colab's local filesystem is ephemeral. The download cell at the end pulls the artifacts to your machine so they survive the runtime.

In [ ]:
final_model_path = MODEL_DIR / "ths_agent_final"
model.save(final_model_path)
if USE_VECNORMALIZE:
    train_env.save(str(MODEL_DIR / "vecnormalize.pkl"))

print(f"Final model saved:   {final_model_path}.zip")
print(f"Best checkpoint:     {MODEL_DIR / 'best_model.zip'}")
print(f"TensorBoard logs:    {TENSORBOARD_LOG}")

## 11. Quick sanity evaluation

One deterministic episode per drive cycle with the trained policy; reports total fuel and final SOC. Uses a fresh `THSEnv` (observations are already normalised inside the env).

In [ ]:
for cycle in ("WLTC", "FTP75", "US06"):
    env = THSEnv(cycle)
    obs, _ = env.reset(seed=SEED)
    fuel_total_g = 0.0
    done = truncated = False
    while not (done or truncated):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, truncated, info = env.step(int(action))
        fuel_total_g += float(info["fuel_rate_gs"]) * env.dt
    print(f"{cycle:6s} | steps={env.idx:5d} | fuel={fuel_total_g:8.2f} g | final SOC={info['soc_pct']:.2f} %")

train_env.close()
eval_env.close()

## 12. (Optional) Download the trained artifacts

In [ ]:
try:
    from google.colab import files
    files.download(str(MODEL_DIR / "ths_agent_final.zip"))
    if USE_VECNORMALIZE:
        files.download(str(MODEL_DIR / "vecnormalize.pkl"))
    if (MODEL_DIR / "best_model.zip").exists():
        files.download(str(MODEL_DIR / "best_model.zip"))
except ImportError:
    print("Not on Colab - artifacts are in", MODEL_DIR)